# IL3.4: Escalabilidad y Sostenibilidad
## Notebook 5: Monitoreo, DevOps y Automatización para Escalabilidad

### Objetivo:
Comprender la importancia de la automatización en DevOps, configurar un AutoScaler dinámico reactivo a la carga del sistema y visualizar la arquitectura integradora final del Resultado de Aprendizaje 3 (RA3).

### Operaciones de Agentes en Producción:
1. **CI/CD Pipelines:** Automatizar tests unitarios y la creación de imágenes Docker del agente al actualizar el repositorio.
2. **Infrastructure as Code (IaC):** Utilizar archivos declarativos (como Terraform) para definir y aprovisionar el cluster de contenedores y los balanceadores de carga.
3. **Auto-scaling:** Incrementar de forma dinámica la cantidad de réplicas de contenedores del agente en base al monitoreo en tiempo real de CPU y memoria.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


### Simulación de Auto-Scaler Reactivo
Implementaremos una clase `AutoScaler` que consulta métricas del sistema (CPU, RAM, peticiones por segundo) y ordena escalar horizontalmente (crear réplicas) o reducir instancias para optimizar costos de computación.


In [ ]:
import random

class MetricsCollector:
    def get_system_metrics(self):
        # Simular lectura de telemetría del contenedor
        return {
            "cpu_usage": random.randint(15, 95),
            "memory_usage": random.randint(25, 90),
            "request_rate": random.randint(5, 80)
        }

class ContainerOrchestrator:
    def __init__(self):
        self.replica_count = 2
        
    def scale_up(self):
        self.replica_count += 1
        print(f"[Orquestador] ↑ ALTA CARGA DETECTADA. Escalando réplicas a: {self.replica_count}")
        
    def scale_down(self):
        if self.replica_count > 1:
            self.replica_count -= 1
            print(f"[Orquestador] ↓ BAJA CARGA DETECTADA. Reduciendo réplicas a: {self.replica_count}")

class AutoScaler:
    def __init__(self, orchestrator):
        self.collector = MetricsCollector()
        self.orchestrator = orchestrator
        
    def evaluate(self):
        metrics = self.collector.get_system_metrics()
        print(f"[Monitoreo] CPU: {metrics['cpu_usage']}% | RAM: {metrics['memory_usage']}% | Peticiones: {metrics['request_rate']} req/s")
        
        # Reglas simples de auto-scaling
        if metrics["cpu_usage"] > 75 or metrics["memory_usage"] > 80:
            self.orchestrator.scale_up()
        elif metrics["cpu_usage"] < 30 and metrics["memory_usage"] < 40:
            self.orchestrator.scale_down()
        else:
            print("[AutoScaler] El sistema está dentro de los rangos de operación estables.")

# Instanciar y ejecutar el escalador 5 veces consecutivas
orch = ContainerOrchestrator()
scaler = AutoScaler(orch)

for check in range(5):
    print(f"\n--- Verificación de AutoScaler {check + 1} ---")
    scaler.evaluate()


### Plantillas Conceptual de DevOps e Infraestructura como Código (IaC)

**1. Configuración de Terraform (AWS ECS Cluster y Service):**
```hcl
# terraform/main.tf
resource "aws_ecs_cluster" "agent_cluster" {
  name = "scalable-agents-cluster"
  
  setting {
    name  = "containerInsights"
    value = "enabled"
  }
}

resource "aws_ecs_service" "agent_service" {
  name            = "wikipedia-agent-service"
  cluster         = aws_ecs_cluster.agent_cluster.id
  task_definition = aws_ecs_task_definition.agent.arn
  desired_count   = 3  # Número base de instancias
}
```

**2. CI/CD Pipeline (GitHub Actions - `.github/workflows/deploy.yml`):**
```yaml
name: Deploy Scalable Agent
on:
  push:
    branches: [ main ]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
    - name: Checkout code
      uses: actions/checkout@v2
    - name: Build Docker Image
      run: docker build -t agent-image:latest .
    - name: Run Unit Tests
      run: pytest tests/
    - name: Push to AWS ECR and Deploy
      run: echo "Desplegando réplicas del agente en el cluster de ECS..."
```


### Proyecto Final RA3 - Arquitectura Integradora
A lo largo de esta unidad (RA3), hemos construido y analizado las capas fundamentales para desplegar agentes a nivel de producción y negocio:

```
┌──────────────────────────────────────────────────────────┐
│                 SISTEMA DE AGENTES IA                    │
├──────────────────────────────────────────────────────────┤
│ IL3.4: Escalabilidad Horizontal y Green Computing        │
├──────────────────────────────────────────────────────────┤
│ IL3.3: Sanitización de Inputs, Seguridad y Ética         │
├──────────────────────────────────────────────────────────┤
│ IL3.2: Trazabilidad, trace_id e Historial JSONL          │
├──────────────────────────────────────────────────────────┤
│ IL3.1: Observabilidad, Logs de Sistema y Rendimiento     │
└──────────────────────────────────────────────────────────┘
```


### Preguntas de Análisis
1. **¿Cuáles son los riesgos asociados a una mala configuración de las políticas de auto-scaling (tanto por exceso de escalamiento como por defecto)?**
2. **¿De qué manera la infraestructura como código (IaC) facilita la recuperación ante desastres en un clúster de producción que aloja agentes inteligentes?**
3. **Describa brevemente cómo interactúan los conceptos de Observabilidad (IL3.1), Trazabilidad (IL3.2), Seguridad (IL3.3) y Escalabilidad (IL3.4) en un agente conversacional desplegado a nivel bancario.**
